# Heart Disease Prediction - ML Zoomcamp Project

## 1. Problem
Predict presence of heart disease (binary classification) from clinical features.

Dataset: UCI Cleveland Heart Disease (cleaned version).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

sns.set_style("whitegrid")
%matplotlib inline

## 2. Load & prepare data

In [ ]:
df = pd.read_csv("data/heart_clean.csv")
print(df.shape)
df.head()

## 3. EDA

In [ ]:
print("Missing values:")
print(df.isnull().sum())
print("\nTarget distribution:")
print(df["target"].value_counts(normalize=True))
print("\nBasic stats:")
df.describe()

In [ ]:
# Target balance
plt.figure(figsize=(5,4))
sns.countplot(x="target", data=df)
plt.title("Heart Disease (0=No, 1=Yes)")
plt.show()

In [ ]:
# Correlation heatmap
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(), annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation")
plt.show()

In [ ]:
# Age distribution by target
plt.figure(figsize=(8,4))
sns.histplot(data=df, x="age", hue="target", bins=20, kde=True)
plt.title("Age distribution by heart disease status")
plt.show()

## 4. Feature Importance (preliminary with Random Forest)

In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

rf_temp = RandomForestClassifier(n_estimators=100, random_state=42)
rf_temp.fit(X, y)

imp = pd.Series(rf_temp.feature_importances_, index=X.columns).sort_values(ascending=False)
plt.figure(figsize=(8,5))
imp.plot(kind="barh")
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.show()
print(imp)

## 5. Train / Test split + Model training

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(X_train.shape, X_test.shape)

In [ ]:
# Logistic Regression
lr_pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(max_iter=1000, random_state=42))
])
lr_pipe.fit(X_train, y_train)
y_pred = lr_pipe.predict(X_test)
y_proba = lr_pipe.predict_proba(X_test)[:,1]
print("Logistic Regression")
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC :", roc_auc_score(y_test, y_proba))
print(classification_report(y_test, y_pred))

In [ ]:
# Random Forest + Hyperparameter tuning
param_grid = {
    "n_estimators": [50, 100, 200],
    "max_depth": [3, 5, 7, None],
    "min_samples_leaf": [1, 2, 4]
}
rf = RandomForestClassifier(random_state=42)
grid = GridSearchCV(rf, param_grid, cv=5, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)
print("Best CV ROC-AUC:", grid.best_score_)

best_rf = grid.best_estimator_
y_pred_rf = best_rf.predict(X_test)
y_proba_rf = best_rf.predict_proba(X_test)[:,1]
print("\nTest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("Test ROC-AUC :", roc_auc_score(y_test, y_proba_rf))
print(classification_report(y_test, y_pred_rf))

## 6. Conclusion

Random Forest with tuned hyperparameters is selected as the final model.
The training logic is exported to  and the model is served via Flask in .